In [ ]:
#KOMÓRKA 1 i 2: PRZYGOTOWANIE DANYCH (GBT VERSION)
!pip install -q pyspark kagglehub gradio

from pyspark.sql import SparkSession
import kagglehub
from pyspark.sql.functions import col, when, month, hour
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline

spark = SparkSession.builder.appName("FlightDelay_GBT").getOrCreate()

# Pobranie danych
path = kagglehub.dataset_download("williamparker20/flight-ontime-reporting-with-weather")
df = spark.read.option("recursiveFileLookup", "true").csv(path, header=True, inferSchema=True)

# 1. Wybór kolumn - DODAJEMY 'Dest' (Lotnisko docelowe)
cols_needed = ["DepDelayMinutes", "Time", "Origin", "Dest", "Carrier",
               "Temperature", "Wind_Speed", "Precipitation",
               "Wind_Gust", "Ice_Accretion_3hr"]

# Czyszczenie
df_clean = df.select(cols_needed).na.fill(0.0, ["Wind_Gust", "Ice_Accretion_3hr"]).dropna()

# 2. Label & Time
df_prepared = df_clean.withColumn("Label", when(col("DepDelayMinutes") > 15, 1.0).otherwise(0.0)) \
                      .withColumn("Month", month(col("Time"))) \
                      .withColumn("Hour", hour(col("Time")))

# BALANSOWANIE KLAS NIE JEST POTRZEBNE DLA GBT W TAKIM STOPNIU

# 3. Indeksowanie (Origin, Carrier ORAZ Dest)
indexer_origin = StringIndexer(inputCol="Origin", outputCol="OriginIndex", handleInvalid="keep")
indexer_dest = StringIndexer(inputCol="Dest", outputCol="DestIndex", handleInvalid="keep") # Nowość
indexer_carrier = StringIndexer(inputCol="Carrier", outputCol="CarrierIndex", handleInvalid="keep")

# 4. Assembler
assembler = VectorAssembler(
    inputCols=["OriginIndex", "DestIndex", "CarrierIndex", "Month", "Hour",
               "Temperature", "Wind_Speed", "Wind_Gust", "Precipitation", "Ice_Accretion_3hr"],
    outputCol="features"
)

train_data, test_data = df_prepared.randomSplit([0.8, 0.2], seed=42)
print("Dane przygotowane pod GBT.")

100%|██████████| 157M/157M [00:01<00:00, 143MB/s]

Extracting files...


Dane przygotowane pod GBT.


In [ ]:
#KOMÓRKA 3: TRENING GBT Z WIĘKSZYM MAXBINS
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Zmieniamy Random Forest na GBT
gbt = GBTClassifier(labelCol="Label",
                    featuresCol="features",
                    maxIter=30,      
                    maxDepth=5,      
                    maxBins=500)     

pipeline = Pipeline(stages=[indexer_origin, indexer_dest, indexer_carrier, assembler, gbt])

print("Rozpoczynam trenowanie Gradient Boosted Trees (To może potrwać 2-4 minuty)...")
model = pipeline.fit(train_data)
print("Model GBT gotowy!")

# Ewaluacja
predictions = model.transform(test_data)
evaluator = BinaryClassificationEvaluator(labelCol="Label")
auc = evaluator.evaluate(predictions)

print(f"\n>>> WYNIK AUC (GBT): {auc:.4f} <<<")

# Ważność cech w GBT
importances = model.stages[-1].featureImportances
print("\nWażność cech:")
print(importances)

Rozpoczynam trenowanie Gradient Boosted Trees (To może potrwać 2-4 minuty)...
Model GBT gotowy!

>>> WYNIK AUC (GBT): 0.7048 <<<

Ważność cech:
(10,[0,1,2,3,4,5,6,7,8],[0.13864455603057968,0.066468178274533,0.16658064296587702,0.23960909397338087,0.19072881553010976,0.09263236529199602,0.01097092205694779,0.022221395645553036,0.07214403023102284])


In [ ]:
#KOMÓRKA 4: FRONTEND GBT
import gradio as gr
from pyspark.sql import Row

origins = model.stages[0].labels
dests = model.stages[1].labels
carriers = model.stages[2].labels

origins.sort()
dests.sort()
carriers.sort()

def predict_gbt(carrier, origin, dest, month, hour, temp, wind, gust, precip, ice):
    try:
        user_data = spark.createDataFrame([
            Row(
                Carrier=carrier,
                Origin=origin,
                Dest=dest,        
                Month=int(month),
                Hour=int(hour),
                Temperature=float(temp),
                Wind_Speed=float(wind),
                Wind_Gust=float(gust),
                Precipitation=float(precip),
                Ice_Accretion_3hr=float(ice)
            )
        ])

        pred = model.transform(user_data)
        prob = pred.select("probability").collect()[0][0][1]

        risk = "EKSTREMALNE" if prob > 0.7 else "WYSOKIE" if prob > 0.5 else "NISKIE"

        return f"Ryzyko: {risk} ({prob:.1%})\nTrasa: {origin} -> {dest} linią {carrier}."

    except Exception as e:
        return f"Błąd: {str(e)}"

iface = gr.Interface(
    fn=predict_gbt,
    inputs=[
        gr.Dropdown(choices=carriers, label="Linia", value=carriers[0]),
        gr.Dropdown(choices=origins, label="Wylot (Origin)", value=origins[0]),
        gr.Dropdown(choices=dests, label="Cel (Dest)", value=dests[0]),
        gr.Slider(1, 12, label="Miesiąc"),
        gr.Slider(0, 23, label="Godzina", value=17),
        gr.Slider(-10, 100, label="Temp (F)"),
        gr.Slider(0, 40, label="Wiatr"),
        gr.Slider(0, 60, label="Porywy"),
        gr.Slider(0, 2, label="Opady"),
        gr.Slider(0, 5, label="Lód")
    ],
    outputs="text",
    title="Zaawansowana Predykcja Opóźnień (Spark GBT)",
    description="Model Gradient Boosted Trees uwzględniający trasę, przewoźnika i pogodę."
)

iface.launch(share=True)

NameError: name 'model' is not defined